# 🌿 মনের কথা — Colab Backend

**Instructions:**
1. Runtime → Change runtime type → **T4 GPU** (free)
2. Fill in `NGROK_TOKEN` and `HF_TOKEN` in Cell 2
3. Run Cell 1 (install), then Cell 2 (start server)
4. Copy the `PUBLIC URL` printed → paste in Streamlit Cloud secrets
5. Keep this tab open while users use your app!

In [ ]:
!pip install unsloth
!pip install -q fastapi uvicorn pyngrok nest_asyncio
!pip install -q huggingface_hub sentencepiece protobuf
print("✅ Done")

# 

In [ ]:
# ════════════════════════════════════════════════════
# 🔑 FILL THESE IN BEFORE RUNNING
HF_TOKEN    = ''#   # HuggingFace token (if model is private)
NGROK_TOKEN = '' #3Aos8uIH50cQNq8w5AmVsYh8XGF_7Zyx3eE24oQCwGtHi2F6d  # <--- PLEASE REPLACE THIS WITH YOUR VALID NGROK_TOKEN from https://ngrok.com → Your Authtoken
# ════════════════════════════════════════════════════

# 1. Install the high-speed transfer library
!pip install hf_transfer

import os
os.environ["UNSLOTH_USE_MODELSCOPE"] = "0"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1" 

from unsloth import FastLanguageModel
import torch

# Configuration
MODEL_ID    = "kazol196295/llama-3.1-8b-bengali-mental-health-counsellor-final"

print("🚀 Loading Bengali Mental Health Counselor...")

# This one command handles the Base Model + Your Adapters automatically
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
    token          = HF_TOKEN,
)

FastLanguageModel.for_inference(model) # Enable 2x faster inference
print("✅ Everything is ready! Your counselor is online.")




🚀 Loading Bengali Mental Health Counselor...
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.


adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Unsloth 2026.4.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Everything is ready! Your counselor is online.


In [4]:



# ── FastAPI ───────────────────────────────────────────────────
import threading, asyncio, json
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from transformers import TextIteratorStreamer

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class ChatRequest(BaseModel):
    topic: str
    message: str

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/chat/stream")
async def chat_stream(req: ChatRequest):
    prompt = (
        f"[INST]\nTopic: {req.topic}\nUser: {req.message}\n"
        f"Respond empathetically as a counselor.\n[/INST]\nCounselor:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    gen_kwargs = dict(
        **inputs,
        max_new_tokens    = 400,
        temperature       = 0.75,
        do_sample         = True,
        top_p             = 0.92,
        repetition_penalty= 1.15,
        pad_token_id      = tokenizer.eos_token_id,
        streamer          = streamer,
    )
    thread = threading.Thread(target=model.generate, kwargs=gen_kwargs)
    thread.start()

    async def token_gen():
        for token in streamer:
            if token:
                yield json.dumps({"token": token}, ensure_ascii=False) + "\n"
                await asyncio.sleep(0)
        yield json.dumps({"done": True}) + "\n"
        thread.join()

    return StreamingResponse(token_gen(), media_type="application/x-ndjson")

# ── ngrok + uvicorn ───────────────────────────────────────────
import uvicorn, nest_asyncio
from pyngrok import ngrok, conf
PORT = 8000
conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()
tunnel = ngrok.connect(PORT, "http")
url    = tunnel.public_url

print()
print("=" * 55)
print(f"  🌐  PUBLIC URL:  {url}")
print("=" * 55)
#print(f'  Streamlit Secrets → BACKEND_URL = "{url}"')
print("  ⚠️  Keep this tab OPEN!")
print("=" * 55)

nest_asyncio.apply()

def run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=PORT)

# Run uvicorn in a separate thread
uvicorn_thread = threading.Thread(target=run_uvicorn)
uvicorn_thread.start()

# To keep the Colab cell alive and show the output, we can use a simple loop
# You might need to manually stop the cell execution when done, or add a stop mechanism.
# For now, this will allow the server to run in the background.
import time
while True:
    time.sleep(1) # Keep the main thread busy so the output is visible, but doesn't block the server


INFO:     Started server process [193]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



  🌐  PUBLIC URL:  https://peremptory-fourthly-argelia.ngrok-free.dev
  ⚠️  Keep this tab OPEN!
INFO:     103.174.23.41:0 - "GET /health HTTP/1.1" 200 OK
INFO:     103.174.23.41:0 - "POST /chat/stream HTTP/1.1" 200 OK


Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

INFO:     103.174.23.42:0 - "GET /health HTTP/1.1" 200 OK
INFO:     103.174.23.42:0 - "POST /chat/stream HTTP/1.1" 200 OK


Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     103.174.23.41:0 - "GET /health HTTP/1.1" 200 OK
INFO:     103.174.23.41:0 - "POST /chat/stream HTTP/1.1" 200 OK


Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     103.174.23.42:0 - "GET /health HTTP/1.1" 200 OK
INFO:     35.197.92.111:0 - "GET /health HTTP/1.1" 200 OK
INFO:     35.197.92.111:0 - "POST /chat/stream HTTP/1.1" 200 OK


Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 